# The Chess Code

## A Data-Driven Analysis of Chess Game Outcomes and Player Skill

**Student Name:** Din Muhammad Rezwoan  
**Dataset:** Chess Game Dataset (Lichess) - https://www.kaggle.com/datasets/datasnaek/chess  
**Source:** Kaggle  
**Tools:** Python, Pandas, NumPy, Matplotlib, Scikit-learn


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay,
                             mean_absolute_error,
                             r2_score)
import kagglehub
import shutil
import os

print("All libraries imported successfully")

All libraries imported successfully


## Section 1 - Load Dataset


In [34]:
# Download dataset from Kaggle
path = kagglehub.dataset_download("datasnaek/chess")
print("Downloaded to:", path)

# Copy games.csv into project /data folder
src = os.path.join(path, 'games.csv')
dst = os.path.join(os.getcwd(), 'data', 'games.csv')

if not os.path.exists(dst):
    shutil.copy(src, dst)
    print("games.csv copied to /data folder")
else:
    print("games.csv already exists in /data folder")

# Load dataset
df = pd.read_csv(dst)
print(f"\nDataset loaded successfully")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Downloaded to: C:\Users\Admin\.cache\kagglehub\datasets\datasnaek\chess\versions\1
games.csv already exists in /data folder

Dataset loaded successfully
Rows: 20058
Columns: 16


## Section 2 - Data Understanding


In [35]:
print("Column Names:")
print(df.columns)
print("\nData Types:")
print(df.dtypes)

Column Names:
Index(['id', 'rated', 'created_at', 'last_move_at', 'turns', 'victory_status',
       'winner', 'increment_code', 'white_id', 'white_rating', 'black_id',
       'black_rating', 'moves', 'opening_eco', 'opening_name', 'opening_ply'],
      dtype='str')

Data Types:
id                    str
rated                bool
created_at        float64
last_move_at      float64
turns               int64
victory_status        str
winner                str
increment_code        str
white_id              str
white_rating      float64
black_id              str
black_rating        int64
moves                 str
opening_eco           str
opening_name          str
opening_ply         int64
dtype: object


In [36]:
print("First 5 rows:")
df.head()

First 5 rows:


,id,rated,created_at,last_move_at,turns,victory_status,winner,increment_code,white_id,white_rating,black_id,black_rating,moves,opening_eco,opening_name,opening_ply
0,TZJHLljE,False,1.504210e+12,1.504210e+12,13,outoftime,white,15+2,bourgris,1500.0,a-00,1191,d4 d5 c4 c6 cxd5 e6 dxe6 fxe6 Nf3 Bb4+ Nc3 Ba5...,D10,Slav Defense: Exchange Variation,5
1,l1NXvwaE,True,1.504130e+12,1.504130e+12,16,resign,black,5+10,a-00,1322.0,skinnerua,1261,d4 Nc6 e4 e5 f4 f6 dxe5 fxe5 fxe5 Nxe5 Qd4 Nc6...,B00,Nimzowitsch Defense: Kennedy Variation,4
2,mIICvQHh,True,1.504130e+12,1.504130e+12,61,mate,white,5+10,ischia,1496.0,a-00,1500,e4 e5 d3 d6 Be3 c6 Be2 b5 Nd2 a5 a4 c5 axb5 Nc...,C20,King's Pawn Game: Leonardis Variation,3
3,kWKvrqYL,True,1.504110e+12,1.504110e+12,61,mate,white,20+0,daniamurashov,1439.0,adivanov2009,1454,d4 d5 Nf3 Bf5 Nc3 Nf6 Bf4 Ng4 e3 Nc6 Be2 Qd7 O...,D02,Queen's Pawn Game: Zukertort Variation,3
4,9tXo1AUZ,True,1.504030e+12,1.504030e+12,95,mate,white,30+3,nik221107,1523.0,adivanov2009,1469,e4 e5 Nf3 d6 d4 Nc6 d5 Nb4 a3 Na6 Nc3 Be7 b4 N...,C41,Philidor Defense,5


In [37]:
print("Numerical Summary:")
df.describe()

Numerical Summary:


,created_at,last_move_at,turns,white_rating,black_rating,opening_ply
count,2.005800e+04,2.005800e+04,20058.000000,19758.000000,20058.000000,20058.000000
mean,1.483617e+12,1.483618e+12,60.465999,1596.480970,1588.831987,4.816981
std,2.850151e+10,2.850140e+10,33.570585,291.478336,291.036126,2.797152
min,1.376772e+12,1.376772e+12,1.000000,784.000000,789.000000,1.000000
25%,1.477548e+12,1.477548e+12,37.000000,1398.000000,1391.000000,3.000000
50%,1.496010e+12,1.496010e+12,55.000000,1566.500000,1562.000000,4.000000
75%,1.503170e+12,1.503170e+12,79.000000,1793.000000,1784.000000,6.000000
max,1.504493e+12,1.504494e+12,349.000000,2700.000000,2723.000000,28.000000


In [38]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df[missing_df['Missing Count'] > 0])

              Missing Count  Missing %
white_rating            300        1.5
opening_eco             601        3.0
opening_name            802        4.0


In [39]:
print("Victory Type Distribution:")
print(df['victory_status'].value_counts())
print("\nWinner Distribution:")
print(df['winner'].value_counts(dropna=False))
print("\nRated vs Casual:")
print(df['rated'].value_counts())


Victory Type Distribution:
victory_status
resign       11147
mate          6325
outoftime     1680
draw           906
Name: count, dtype: int64

Winner Distribution:
winner
white    10001
black     9107
draw       950
Name: count, dtype: int64

Rated vs Casual:
rated
True     16155
False     3903
Name: count, dtype: int64


In [40]:
print(df.isnull().sum())
print("\nTotal rows:", len(df))

id                  0
rated               0
created_at          0
last_move_at        0
turns               0
victory_status      0
winner              0
increment_code      0
white_id            0
white_rating      300
black_id            0
black_rating        0
moves               0
opening_eco       601
opening_name      802
opening_ply         0
dtype: int64

Total rows: 20058


### Data Understanding Summary

- **Rows:** 20,058 games across 16 columns.
- `created_at` and `last_move_at` require timestamp conversion.
- `winner` uses "draw" for tied games (no missing values).
- **Missing Data:**
  - `white_rating` - 300 rows missing.
  - `opening_eco` - 601 rows missing.
  - `opening_name` - 802 rows missing.
- `victory_type` is imbalanced (~60% resignations).
- `increment_code` (e.g., "5+3") requires splitting.

Missing rows in `white_rating` will be dropped. Missing opening data will be filled with "Unknown".


## Section 3 - Data Cleaning


**Step 1 - Compute Estimated Game Duration:**
Calculate game duration using `time_base` and `time_increment` extracted from `increment_code`. Drop the raw string and unneeded timestamp columns.


In [41]:
# Split increment_code into two usable columns first
df['time_base'] = df['increment_code'].str.split('+').str[0].astype(float)
df['time_increment'] = df['increment_code'].str.split('+').str[1].astype(float)

# Estimate game duration using turns and time control
# Logic: both players use their base time + each move adds increment seconds
df['game_duration_mins'] = (
    (df['time_base'] * 2) + (df['turns'] * df['time_increment'] / 60)
).round(2)

# Drop the unusable timestamp columns and original increment_code
df.drop(columns=['created_at', 'last_move_at', 'increment_code'], inplace=True)

print("Done")
print(df[['time_base', 'time_increment', 'turns', 'game_duration_mins']].head(10))

Done
   time_base  time_increment  turns  game_duration_mins
0       15.0             2.0     13               30.43
1        5.0            10.0     16               12.67
2        5.0            10.0     61               20.17
3       20.0             0.0     61               40.00
4       30.0             3.0     95               64.75
5       10.0             0.0      5               20.00
6       10.0             0.0     33               20.00
7       15.0            30.0      9               34.50
8       15.0             0.0     66               30.00
9       10.0             0.0    119               20.00


**Step 2 - Drop Rows with Missing Values:**
Drop rows with missing data in `white_rating`, `opening_eco`, and `opening_name` to maintain dataset accuracy.


In [42]:
before = len(df)

df.dropna(inplace=True)

after = len(df)
print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Rows dropped: {before - after}")
print(f"\nRemaining null values:")
print(df.isnull().sum().sum())

Rows before: 20058
Rows after: 18380
Rows dropped: 1678

Remaining null values:
0


**Step 3 - Remove Duplicates:** Drop duplicate rows based on the `id` column.


In [44]:
# Step 3 — Remove duplicate games
before = len(df)
df.drop_duplicates(subset='id', inplace=True)
print(f"Duplicates removed: {before - len(df)}")
print(f"Rows remaining: {len(df)}")

Duplicates removed: 802
Rows remaining: 17578


**Step 4 - Drop Moves Column**


In [45]:
# Step 4 — Drop moves column
df.drop(columns=['moves'], inplace=True)
print("Final shape:", df.shape)

Final shape: (17578, 15)


## Section 4 - Feature Engineering


In [46]:
# Feature 1 — Rating gap (positive = white stronger, negative = black stronger)
df['rating_gap'] = df['white_rating'] - df['black_rating']

# Feature 2 — Average rating (overall skill level of the game)
df['avg_rating'] = ((df['white_rating'] + df['black_rating']) / 2).round(0)

# Feature 3 — Rating tier
df['rating_tier'] = pd.cut(
    df['avg_rating'],
    bins=[0, 1200, 1500, 1800, 2100, 3000],
    labels=['Beginner', 'Intermediate', 'Advanced', 'Expert', 'Master']
)

# Feature 4 — Time control category
df['time_control_category'] = pd.cut(
    df['time_base'],
    bins=[0, 3, 5, 10, 9999],
    labels=['Bullet', 'Blitz', 'Rapid', 'Classical']
)

# Feature 5 — Opening family
df['opening_family'] = df['opening_name'].str.split(':').str[0].str.strip()

# Feature 6 — Upset (underdog won)
df['is_upset'] = (
    ((df['rating_gap'] > 0) & (df['winner'] == 'black')) |
    ((df['rating_gap'] < 0) & (df['winner'] == 'white'))
)

print("Features created successfully")
print(df[['rating_gap', 'avg_rating', 'rating_tier', 
          'time_control_category', 'opening_family', 'is_upset']].head(10))

Features created successfully
   rating_gap  avg_rating   rating_tier time_control_category  \
0       309.0      1346.0  Intermediate             Classical   
1        61.0      1292.0  Intermediate                 Blitz   
2        -4.0      1498.0  Intermediate                 Blitz   
3       -15.0      1446.0  Intermediate             Classical   
4        54.0      1496.0  Intermediate             Classical   
5       248.0      1126.0      Beginner                 Rapid   
6        97.0      1472.0  Intermediate                 Rapid   
7      -695.0      1760.0      Advanced             Classical   
8        47.0      1416.0  Intermediate             Classical   
9       172.0      1295.0  Intermediate                 Rapid   

           opening_family  is_upset  
0            Slav Defense     False  
1     Nimzowitsch Defense      True  
2        King's Pawn Game      True  
3       Queen's Pawn Game      True  
4        Philidor Defense     False  
5        Sicilian Defense 